In [ ]:
# transformers-01: toy_transformer GPU
# -----------------------------------------------
# A minimal, easy-to-read Transformer in PyTorch
# that learns to reverse a short character string.
# Focuses on clarity, not speed.

import sys
from pathlib import Path

# Add pytutorials the path (one level up from notebooks/) to sys.path
sys.path.append(str(Path.cwd().parent))

# Imports
import random
import torch
from pytutorials.data.reverse_string import get_dataloaders, decode, VOCAB_SIZE
from pytutorials.models.transformer import TinyTransformer
from pytutorials.training.basic import create_optimizer_and_loss, run_epoch, make_padding_mask
from pytutorials.utils.gpu import get_device, optimize_model


In [ ]:
# Check for available device
device = get_device()
print("Using", device)

In [ ]:
# Set up data loaders
train_loader, val_loader = get_dataloaders(batch_size=64)

In [ ]:
# Initialize model, loss, and optimizer
model                = TinyTransformer(vocab_size=VOCAB_SIZE)
model = optimize_model(model, device=device, dtype=None, compile_model=False)
print("Model parameters:", sum(p.numel() for p in model.parameters()))

criterion, optimizer = create_optimizer_and_loss(model, lr=1e-3)

In [ ]:
# Training loop with qualitative evaluation
from pytutorials.data.reverse_string import ReverseDataset
from torch.utils.data import DataLoader

# Set both seeds manually (approximate behavior)
random.seed(42)
torch.manual_seed(42)

for epoch in range(10):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, train=True, device=device)
    val_loss,   val_acc   = run_epoch(model, val_loader,   criterion, train=False, device=device)
    print(f"Epoch {epoch:2d} | "
          f"train loss {train_loss:.3f}, acc {train_acc:.2%} | "
          f"val loss {val_loss:.3f}, acc {val_acc:.2%}")

    # Qualitative example
    if epoch % 1 == 0:
        sample_batch = next(iter(DataLoader(ReverseDataset("val", n_samples=128), batch_size=8, shuffle=True)))
        sample_inp, sample_tgt = sample_batch
        
        sample_inp = sample_inp.to(device)
        sample_tgt = sample_tgt.to(device)
        mask       = make_padding_mask(sample_inp).to(device)
        pred       = model(sample_inp, mask).argmax(-1)
        
        print(" Example predictions:")
        for i in range(min(5, len(sample_inp))):
            print(f"   input : {decode(sample_inp[i].tolist())}")
            print(f"   target: {decode(sample_tgt[i].tolist())}")
            print(f"   pred  : {decode(pred[i].tolist())}")
            print("   " + "-"*30)